In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Lesson Learn

## StateMachine
This idea using statemachine idea to iterative run until we met the end state

## State: with Enum
This idea serves as a blueprint what we can move to which state

## Data: with pydantic.BaseModel
This idea serves as a data spec that will be used in each state

## Method: with ABC @abstractmethod
This idea serves as a contract of each class

# New State Machine Idea

In [2]:
from broinsight.statemachines.simple_statemachine import SimpleStateMachine, simple_state, BaseSimpleContext, BaseSimpleState, SimpleStateRegistry

In [3]:
from broinsight.core.llm import LocalOpenAI, UserMessage, AIMessage
from broinsight.prompt_hub import PromptHub

In [4]:
print(PromptHub().quick_chat)

# PERSONA

You're Andy who is the best bro in the world. You're always chill and supportive.

# INSTRUCTION

- understand {USER_INPUT}
- always respond your message in bro-like manner

# CONDITION

- your answer will always be concise
- if {USER_INPUT} has the intent indicating a user needs a long, detailed answer like: `explain in detail`, `explain it step by step`, you can answer it accordingly



In [5]:
from enum import Enum
from typing import Any
from pydantic import Field

class SimpleState(Enum):
    USER_INPUT = "user_input"
    ROUTER = "router"
    CHAT = "chat"
    COMPLETE = "complete"

class SimpleContext(BaseSimpleContext):
    user_input:Any = Field(default=None)
    input_state:Any = Field(default=None)
    output_state:Any = Field(default=None)
    messages:list = Field(default_factory=list)

@simple_state(SimpleState.USER_INPUT)
class UserInputState(BaseSimpleState):
    def next_state(self): return SimpleState.ROUTER
    def run(self, context: SimpleContext):
        context.user_input = input("Enter your input: ")
        return self.next_state()

@simple_state(SimpleState.ROUTER)
class RouterState(BaseSimpleState):
    def next_state(self, user_input:str):
        if user_input.lower().startswith("/exit"):
            return SimpleState.COMPLETE
        return SimpleState.CHAT
    def run(self, context: SimpleContext): return self.next_state(context.user_input)
    
@simple_state(SimpleState.CHAT)
class ChatState(BaseSimpleState):
    def __init__(self): self.llm = LocalOpenAI()
    def next_state(self): return SimpleState.USER_INPUT
    def run(self, context: SimpleContext):
        messages = context.messages
        messages.append(UserMessage(content=context.user_input))
        response = self.llm.run(PromptHub().quick_chat, messages)
        messages.append(AIMessage(content=response.content))
        context.messages = messages
        return self.next_state()

In [6]:
SimpleStateRegistry._states

{'USER_INPUT': __main__.UserInputState,
 'ROUTER': __main__.RouterState,
 'CHAT': __main__.ChatState}

In [7]:
SimpleStateRegistry.state_graph()

{'USER_INPUT': ['ROUTER'],
 'ROUTER': ['CHAT', 'COMPLETE'],
 'CHAT': ['USER_INPUT']}

In [8]:
SimpleStateRegistry.to_mermaid(save_path="./flow.md", direction="TB")

'flowchart TB\n    USER_INPUT --> ROUTER\n    ROUTER -.-> CHAT\n    ROUTER -.-> COMPLETE\n    CHAT --> USER_INPUT'

In [9]:
state_machine = SimpleStateMachine(SimpleState.USER_INPUT, SimpleState.COMPLETE)
result = state_machine.run(SimpleContext())

In [10]:
result.execution_trace

[{'state': 'USER_INPUT', 'next_state': 'ROUTER'},
 {'state': 'ROUTER', 'next_state': 'COMPLETE'}]

In [11]:
result.messages

[]

In [12]:
SimpleStateRegistry.clear_all_states()

In [13]:
SimpleStateRegistry._states

{}

In [14]:
import seaborn as sns
import duckdb
# tips = sns.load_dataset('tips')
duckdb.register('tips', sns.load_dataset('tips'))

In [15]:
duckdb.execute("DESCRIBE tips;").df()

,column_name,column_type,null,key,default,extra
0,total_bill,DOUBLE,YES,None,None,None
1,tip,DOUBLE,YES,None,None,None
2,sex,"ENUM('Male', 'Female')",YES,None,None,None
3,smoker,"ENUM('Yes', 'No')",YES,None,None,None
4,day,"ENUM('Thur', 'Fri', 'Sat', 'Sun')",YES,None,None,None
5,time,"ENUM('Lunch', 'Dinner')",YES,None,None,None
6,size,BIGINT,YES,None,None,None


In [16]:
duckdb.execute("DESCRIBE tips;").fetchall()

[('total_bill', 'DOUBLE', 'YES', None, None, None),
 ('tip', 'DOUBLE', 'YES', None, None, None),
 ('sex', "ENUM('Male', 'Female')", 'YES', None, None, None),
 ('smoker', "ENUM('Yes', 'No')", 'YES', None, None, None),
 ('day', "ENUM('Thur', 'Fri', 'Sat', 'Sun')", 'YES', None, None, None),
 ('time', "ENUM('Lunch', 'Dinner')", 'YES', None, None, None),
 ('size', 'BIGINT', 'YES', None, None, None)]

In [17]:
duckdb.description()

[('column_name', 'STRING', None, None, None, None, None),
 ('column_type', 'STRING', None, None, None, None, None),
 ('null', 'STRING', None, None, None, None, None),
 ('key', 'STRING', None, None, None, None, None),
 ('default', 'STRING', None, None, None, None, None),
 ('extra', 'STRING', None, None, None, None, None)]

In [18]:
# Show all tables
result = duckdb.execute("SHOW TABLES").fetchall()
print(f"Number of tables: {len(result)}")

# Or get count directly
count = duckdb.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'main'").fetchall()
print(f"Table count: {count}")


Number of tables: 1
Table count: [(1,)]


In [19]:
result

[('tips',)]

In [20]:
count

[(1,)]

In [ ]:
from broinsight.statemachines.utils import get_state_str, get_return_values_ast
class StateGroup:
    def __init__(self, name: str = "default"):
        self.name = name
        self._states = {}
    
    def state(self, state_name):
        """Instance-based decorator like FastAPI's @app.route"""
        def decorator(cls):
            state_name_str = get_state_str(state_name)
            if state_name_str not in self._states:
                self._states[state_name_str] = cls
            else:
                print(f"Already registered in {self.name}: {state_name_str}")
            return cls
        return decorator
    
    def get(self, state_name):
        return self._states[get_state_str(state_name)]()
    
    def state_graph(self):
        transitions = {}
        for k, v in self._states.items():
            returns = get_return_values_ast(v.next_state)
            transitions[k] = returns
        return transitions
    
    def create_machine(self, start_state, end_state):
        return GroupStateMachine(self, start_state, end_state)

class GroupStateMachine:
    def __init__(self, state_group: StateGroup, start_state, end_state):
        self.state_group = state_group
        self.start_state = start_state
        self.end_state = get_state_str(end_state)
    
    def run(self, context):
        current_state = get_state_str(self.start_state)
        
        while current_state != self.end_state:
            state_instance = self.state_group.get(current_state)
            next_state = state_instance.run(context)
            next_state = get_state_str(next_state)
            context.execution_trace.append({
                "current_state": current_state,
                "next_state": next_state
            })
            current_state = next_state
            
        return context


In [32]:
# Create separate state groups
chat_app = StateGroup("chat_workflow")
data_app = StateGroup("data_workflow")

# Register states to specific groups
@chat_app.state(SimpleState.USER_INPUT)
class ChatUserInput(BaseSimpleState):
    def next_state(self): return SimpleState.COMPLETE
    def run(self, context): 
        return self.next_state()

# @chat_app.state(SimpleState.COMPLETE)

@data_app.state(SimpleState.USER_INPUT)  
class DataUserInput(BaseSimpleState):
    def next_state(self): return SimpleState.COMPLETE
    def run(self, context):
        return self.next_state()

# Create separate machines
chat_machine = chat_app.create_machine(SimpleState.USER_INPUT, SimpleState.COMPLETE)
data_machine = data_app.create_machine(SimpleState.USER_INPUT, SimpleState.COMPLETE)

# Run independently
chat_result = chat_machine.run(SimpleContext())
data_result = data_machine.run(SimpleContext())


In [33]:
chat_result

SimpleContext(execution_trace=[{'current_state': 'USER_INPUT', 'next_state': 'COMPLETE'}], user_input=None, input_state=None, output_state=None, messages=[])

In [36]:
data_result

SimpleContext(execution_trace=[{'current_state': 'USER_INPUT', 'next_state': 'COMPLETE'}], user_input=None, input_state=None, output_state=None, messages=[])

In [34]:
chat_app._states

{'USER_INPUT': __main__.ChatUserInput}

In [35]:
data_app._states

{'USER_INPUT': __main__.DataUserInput}

In [11]:
from broinsight.statemachines.group_statemachine import StateGroup, GroupStateMachine, CompositeStateGroup
from broinsight.statemachines.simple_statemachine import BaseSimpleState, BaseSimpleContext, SimpleStateRegistry, get_state_str
from broinsight.prompt_hub import PromptHub

router_flow = StateGroup("router_workflow")
sql_flow = StateGroup("sql_workflow")
chat_flow = StateGroup("chat_workflow")

In [12]:
from broinsight.utils.data_catalog import DataCatalog
import seaborn as sns

catalog = DataCatalog()
catalog.register("tips", sns.load_dataset('tips'))

In [ ]:
from enum import Enum
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any
from broinsight.core.llm import LocalOpenAI, UserMessage, AIMessage, ModelResponse
from broinsight.utils.parse_string import parse_sql, parse_json

class ComplexState(Enum):
    USER_INPUT = "user_input"
    SQL = "sql"
    CHAT = "chat"
    COMPLETE = "complete"

class RouterContext(BaseModel):
    route:str = Field(default=None)
    response:ModelResponse = Field(default=None)

class ChatContext(BaseModel):
    messages:List[Dict[str, Any]] = Field(default_factory=list)
    response:ModelResponse = Field(default=None)

class SQLContext(BaseModel):
    sql:str = Field(default=None)
    catalog:Any = Field(default=catalog)
    data:Any = Field(default=None)
    response:ModelResponse = Field(default=None)

class ComplexContext(BaseModel):
    user_input:str
    router:RouterContext = Field(default_factory=RouterContext)
    chat:ChatContext = Field(default_factory=ChatContext)
    sql:SQLContext = Field(default_factory=SQLContext)
    execution_trace:List[Dict[str, Any]] = Field(default_factory=list)

@router_flow.register(ComplexState.USER_INPUT)
class RouterState(BaseSimpleState):
    def next_state(self, user_input:str):
        if user_input.lower().startswith("/query"):
            return ComplexState.SQL
        return ComplexState.CHAT
    def run(self, context:ComplexContext):
        return self.next_state(context.user_input)
    
@chat_flow.register(ComplexState.CHAT)
class ChatState(BaseSimpleState):
    def __init__(self): self.llm = LocalOpenAI()
    def next_state(self): return ComplexState.COMPLETE
    def get_content(self, context: ComplexContext):
        user_input = context.user_input
        content = []
        if context.sql.data is not None:
            content.append(f"CONTEXT:\n\n{context.sql.data.to_string()}\n\n")
        content.append(f"USER_INPUT:\n\n{user_input}\n\n")
        return "\n".join(content)
    def run(self, context: ComplexContext):
        messages = context.chat.messages
        content = self.get_content(context)
        response = self.llm.run(PromptHub().quick_chat, messages+[UserMessage(content=content)])
        messages.append(UserMessage(content=context.user_input))
        messages.append(AIMessage(content=response.content))
        context.chat.messages = messages
        context.chat.response = response
        return self.next_state()
    
@sql_flow.register(ComplexState.SQL)
class SQLState(BaseSimpleState):
    def __init__(self): self.llm = LocalOpenAI()
    def next_state(self): return ComplexState.CHAT
    def get_content(self, context:ComplexContext):
        metadata = context.sql.catalog.query("DESCRIBE tips;").loc[:, ['column_name', 'column_type']]
        content = [f"METADATAS: TABLE name's tips\n\n{metadata.to_string()}\n\n"]
        content.append(f"USER_INPUT:\n\n{context.user_input}\n\n")
        return "\n".join(content)
    def run(self, context: ComplexContext):
        messages = context.chat.messages
        # user_input = UserMessage(context.user_input)
        content = self.get_content(context)
        response = self.llm.run(PromptHub().generate_sql, messages+[UserMessage(content=content)])
        context.sql.response = response
        sql_query = parse_sql(response.content)
        context.sql.sql = sql_query
        query_result = context.sql.catalog.query(sql_query)
        context.sql.data = query_result
        return self.next_state()

In [14]:
# Combine all groups
combined_app = CompositeStateGroup("combined", router_flow, chat_flow, sql_flow)

# Create machine with all states available
machine = combined_app.create_machine(ComplexState.USER_INPUT, ComplexState.COMPLETE)
# content = "Write me a SQL bro."
content = "/query Which gender tips the most?"
# content = "Hi there!"
# content = "I wanna which customer paid most in 2023? Could you query it for me?"
context = ComplexContext(user_input=content)
result = machine.run(context)
result.execution_trace

[{'current_state': 'USER_INPUT', 'next_state': 'SQL'},
 {'current_state': 'SQL', 'next_state': 'CHAT'},
 {'current_state': 'CHAT', 'next_state': 'COMPLETE'}]

In [15]:
result.chat.messages

[{'role': 'user', 'content': '/query Which gender tips the most?'},
 {'role': 'assistant',
  'content': 'Bro, based on the data, the Male group is topping the tip charts with $485.07 in total. 🚀'}]

In [16]:
context.execution_trace

[{'current_state': 'USER_INPUT', 'next_state': 'SQL'},
 {'current_state': 'SQL', 'next_state': 'CHAT'},
 {'current_state': 'CHAT', 'next_state': 'COMPLETE'}]

In [17]:
print(context.sql.sql)

SELECT 
    sex,
    SUM(tip) AS total_tip
FROM 
    tips
GROUP BY 
    sex
ORDER BY 
    total_tip DESC
LIMIT 1;


In [18]:
catalog.query(context.sql.sql)

,sex,total_tip
0,Male,485.07
